# Lecture: How Pictures Become Words for a Language Model

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Trace the path pixels take: image → encoder → projection → visual prefix → Transformer → answer, naming each tensor shape.
- Explain prefix fusion (§5): why gluing visual tokens BEFORE the question words conditions every answer on the image.
- Read the system map (§9) and predict the baselines: text-only ≈ 0.20 (blind), image-only ≈ 0.82 joint but 0.000 on color, frozen costs ≈ 9 points.


## The big idea

A language model only understands sequences of word-vectors (lists of numbers, one per word). A picture is a grid of pixels, so the model cannot read it directly. The trick is translation in three hops. First the **encoder** (a small vision network) summarizes the 3×24×24 image into one 256-number summary. Then the **projection** (one matrix multiply) translates that summary into K fake word-vectors called the **visual prefix**. Finally **prefix fusion** glues those photo-words IN FRONT of the real question words, so the Transformer reads one sentence shaped like [photo-words | question-words] and the **answer head** picks 1 of 24 answers. The training signal is plain **cross-entropy (CE)**: reward probability on the true answer. Two **baselines** prove both halves matter: hide the image (**text-only**) and the model guesses blindly; hide the question (**image-only**) and shared images become ambiguous. Lock the encoder's eyes at random (**frozen**) and the rest must cope with bad features — accuracy drops even though everything else still trains.


In [ ]:
import numpy as np  # bring in numpy for shape arithmetic on the wiring demo
import matplotlib  # bring in matplotlib core to pin the headless backend
matplotlib.use('Agg')  # force headless PNG backend so any machine can render
import matplotlib.pyplot as plt  # bring in pyplot for the wiring diagram
import matplotlib.patches as patches  # bring in patches for rounded stage boxes
np.random.seed(19)  # fix the seed so the diagram layout is identical every run
stages = ['image\n3x24x24', 'CNN encoder\npixels -> 256', 'projection\n256 -> 4x64', 'visual prefix\n4 tokens', 'Transformer + Q\n-> 1 answer']  # five pipeline stage labels
subs = ['what you see', 'eyes: find shapes', 'translator to words', 'photo becomes words', 'reads photo-words + Q']  # plain caption under each box
shapes = ['1728 px', '256 numbers', '4 x 64', 'glued BEFORE Q', '24 answers']  # tensor-shape note above each box
print('shapes: image 1728 px -> enc 256 -> prefix 4x64 -> answer 1 of 24')  # print the wiring contract in one line
print('K=4 visual tokens plus Q=4 question tokens gives a sequence of length 8')  # print the fused length 4 + 4 = 8
fig, ax = plt.subplots(figsize=(11, 3.2))  # create a wide canvas for five stages
ax.set_xlim(0, 10)  # fix horizontal room for boxes plus arrows
ax.set_ylim(0, 2.3)  # fix vertical room for boxes plus captions
ax.axis('off')  # hide axes because this is a diagram not data
ax.set_title('VLM wiring: image -> encoder -> projection -> prefix -> answer', fontsize=11)  # title the sketch
for i in range(5):  # walk the five stages left to right
    x = 0.3 + i * 1.95  # compute the left edge of box i
    box = patches.FancyBboxPatch((x, 0.6), 1.6, 0.9, boxstyle='round,pad=0.02', facecolor='#dbeafe', edgecolor='#1d4ed8')  # rounded blue box for this stage
    ax.add_patch(box)  # draw the box on the canvas
    ax.text(x + 0.8, 1.12, stages[i], ha='center', va='center', fontsize=8, weight='bold')  # bold stage name inside the box
    ax.text(x + 0.8, 0.38, subs[i], ha='center', va='top', fontsize=7, style='italic')  # plain caption under the box
    ax.text(x + 0.8, 1.72, shapes[i], ha='center', va='bottom', fontsize=7, color='#b45309')  # gold shape note above the box
    _ = i  # touch the loop var so every line keeps its comment habit
for i in range(4):  # draw one arrow between each pair of neighbours
    x = 0.3 + i * 1.95  # recompute the left edge of box i
    ax.annotate('', xy=(x + 1.95, 1.05), xytext=(x + 1.6, 1.05), arrowprops=dict(arrowstyle='->', color='black', lw=1.5))  # arrow from this box to the next
plt.tight_layout()  # snug the layout so labels are not clipped
plt.show()  # render the wiring diagram


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 builds the photo-words, §5 glues them to the question, §6–§7 read the scoreboard, §8 is the training signal.

| Term | One-liner | Section |
|---|---|---|
| Encoder | Small CNN: pixels → one 256-number summary | §4 |
| Projection | One matrix: 256 summary → K×d photo-words | §4 |
| Visual prefix | The K fake word-vectors glued BEFORE the question | §4 |
| Prefix fusion | Concatenating [visual \| question] so attention mixes sight + words | §5 |
| Answer head | Final layer: pooled state → 24 answer scores | §5 |
| Joint accuracy | Fraction correct over all validation tasks mixed | §6 |
| Text-only baseline | Vision zeroed: the blind-guess control (≈ 0.205) | §6 |
| Image-only baseline | Question zeroed: vision without the task (≈ 0.822 joint, 0.000 color) | §6 |
| Frozen encoder | Eyes locked at random: only the rest learns (≈ 9-point cost) | §7 |
| Cross-entropy (CE) | −log(probability of the true answer): the training loss | §8 |


### §4 Encoder, projection, and the visual prefix

**In one sentence: the encoder summarizes the image into 256 numbers, the projection translates them with one matrix multiply into K visual tokens, and those tokens are the visual prefix the language model reads as photo-words.**

The lab's numbers: 3×24×24 = 1728 pixels → CNN → 256 summary → projection → 4 tokens × 64 dims. The projection is the whole bridge — without it the LM's word-embeddings (width 64) cannot even touch vision features (width 256). Watch it: a toy 4-pixel image times a 4×2 matrix gives one 2-number photo-word; its dot product answers 'shape' more than 'color' for this toy, and zeroing vision (text-only) zeroes both matches.


In [ ]:
import numpy as np  # reuse numpy for the toy projection math
import matplotlib.pyplot as plt  # reuse pyplot for the token bars
np.random.seed(19)  # fix the seed for reproducibility
img = np.array([0.9, 0.1, 0.2, 0.8])  # define a fake 4-pixel image standing in for 3x24x24
W = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.0], [0.0, 0.5]])  # define the toy projection matrix 4 pixels -> 2 dims
tok = img @ W  # project: one matrix multiply turns the photo into numbers
print('visual token:', np.round(tok, 3), '(expect [1. 0.5]: 0.9+0.1 and 0.1+0.4)')  # print the photo-becomes-numbers result
q_shape = np.array([1.0, 0.0])  # define the toy embedding of the question word shape
q_color = np.array([0.0, 1.0])  # define the toy embedding of the question word color
s_shape = float(tok @ q_shape)  # dot-match of the visual token with the shape question
s_color = float(tok @ q_color)  # dot-match of the visual token with the color question
print('match with shape:', round(s_shape, 3), 'vs color:', round(s_color, 3))  # print 1.0 vs 0.5: same photo, different question
print('without vision both matches are 0.0 -> pure guess (the text-only control)')  # print why zeroed prefix carries no signal
print('lab shapes: 1728 px -> enc 256 -> prefix 4x64 -> fused length 8')  # print the real lab wiring beside the toy
fig, ax = plt.subplots(figsize=(6, 4))  # make a figure for the token bars
ax.bar(['dim0 (shape-y)', 'dim1 (color-y)'], tok, color=['#ef4444', '#3b82f6'])  # draw the two photo-word components
ax.set_ylim(0, 1.2)  # fix the range with headroom
ax.set_ylabel('token value')  # label the y-axis
ax.set_title('Toy visual token [1.0, 0.5]: projection output')  # title naming the vocabulary term
ax.grid(True, alpha=0.3, axis='y')  # add a light grid for reading values
plt.tight_layout()  # snug the layout
plt.show()  # render the bars


### §5 Prefix fusion and the answer head

**In one sentence: prefix fusion concatenates [K visual tokens | 4 question tokens] into one length-8 sequence, and the answer head maps the pooled question positions to 24 answer scores.**

Why position matters: the Transformer attends over the whole sequence, so every question slot can look back at every photo-word — generation is conditioned on the image by construction. The lab mean-pools the 4 question positions (not the visual ones) then applies one Linear layer to 24 classes. Watch it: the strip diagram shows the glued order, then tiny numbers walk through pooling to logits.


In [ ]:
import numpy as np  # reuse numpy for the tiny fusion math
import matplotlib.pyplot as plt  # reuse pyplot for the concat strip
np.random.seed(19)  # fix the seed for reproducibility
vis = np.array([[1.0, 0.5], [0.8, 0.2], [0.3, 0.9], [0.6, 0.6]])  # fake visual prefix: K=4 tokens x d=2
txt = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5], [0.2, 0.2]])  # fake question embeddings: Q=4 tokens x d=2
seq = np.concatenate([vis, txt], axis=0)  # fuse: glue photo-words BEFORE question words along time
print('vis', vis.shape, '+ txt', txt.shape, '-> fused', seq.shape, '(4+4=8 rows)')  # print the concat shapes
pooled = txt.mean(axis=0)  # pool: average the 4 question positions like the lab does
print('pooled question state:', np.round(pooled, 3), '(mean of the 4 txt rows)')  # print the pooled vector
logits = np.array([pooled[0] * 2.0, pooled[1] * 2.0, 0.1])  # fake head: 3-way scores from the pooled state
print('toy logits:', np.round(logits, 3), '-> winner class', int(np.argmax(logits)))  # print ranking-only scores
print('lab head: pooled (64,) -> Linear -> 24 answer logits (1-token answer)')  # print the real head contract
fig, ax = plt.subplots(figsize=(8, 2.4))  # make a flat strip for the concat order
ax.set_xlim(0, 8)  # fix room for 8 token slots
ax.set_ylim(0, 1)  # fix vertical room
ax.axis('off')  # hide axes: this is a diagram
ax.set_title('Prefix fusion: [V0 V1 V2 V3 | Q0 Q1 Q2 Q3] -> pool Q -> 24 logits')  # title the order rule
for i in range(8):  # draw one slot per sequence position
    c = '#dbeafe' if i < 4 else '#fef3c7'  # color blue = vision, amber = question
    ax.add_patch(plt.Rectangle((i + 0.05, 0.25), 0.9, 0.5, facecolor=c, edgecolor='black'))  # draw the slot box
    lab = ('V' + str(i)) if i < 4 else ('Q' + str(i - 4))  # pick the slot label by half
    ax.text(i + 0.5, 0.5, lab, ha='center', va='center', fontsize=9)  # label the slot
plt.tight_layout()  # snug the layout
plt.show()  # render the strip


### §6 Baselines and joint accuracy

**In one sentence: joint accuracy is the fraction correct over every validation task mixed together, and the two baselines — text-only (≈ 0.205, vision zeroed) vs image-only (≈ 0.822 joint but 0.000 on color) — prove vision is necessary AND the question is necessary.**

Read the split, not just the joint: shape and color questions share the SAME images and differ only by the question words, so image-only can at best repeat one side — the lab's 0.000 on color exposes the blindness hiding inside a decent-looking 0.822 joint. Text-only near 0.205 is chance-level guessing from question priors. Watch it: the three bars side by side.


In [ ]:
import numpy as np  # reuse numpy for the baseline arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the baseline bars
np.random.seed(19)  # fix the seed for reproducibility
full, txt, im = 0.999, 0.205, 0.822  # representative lab joints: full vs text-only vs image-only
print('full VLM joint:', full, '| text-only:', txt, '| image-only:', im)  # print the three-way comparison
print('vision dividend: full - text-only =', round(full - txt, 3), '(pixels carry the answer)')  # print what sight buys
print('question dividend: full - image-only =', round(full - im, 3), '(joint; color 1.0 -> 0.0)')  # print what the question buys
print('blind spot: image-only color = 0.0 despite joint 0.822 (shared images, no question)')  # print the per-task exposure
fig, ax = plt.subplots(figsize=(6, 4))  # make a figure for the baseline bars
ax.bar(['full VLM', 'text-only', 'image-only'], [full, txt, im], color=['#16a34a', '#9ca3af', '#f59e0b'])  # draw one bar per model
ax.set_ylim(0, 1.1)  # fix the accuracy range with headroom
ax.set_ylabel('joint accuracy')  # label the y-axis
ax.set_title('Baselines: vision matters, question matters (color 0.0)')  # title the lesson
ax.grid(True, alpha=0.3, axis='y')  # add a light grid for reading values
plt.tight_layout()  # snug the layout
plt.show()  # render the bars


### §7 Frozen encoder — what training the eyes buys

**In one sentence: a frozen encoder locks the CNN at random weights so only the Transformer and head learn, isolating what trained vision features are worth: about 9 points of joint accuracy (trained ≈ 0.999 vs frozen ≈ 0.910).**

Freezing is a capacity-vs-features test: the frozen model keeps the same prefix length K=4 and the same Transformer, so any gap is pure feature quality, not slots. The frozen net still learns a linear-ish readout on random features (well above chance), just a worse one. Watch it: trained vs frozen bars with the ≈ 0.089 gap annotated.


In [ ]:
import numpy as np  # reuse numpy for the frozen gap math
import matplotlib.pyplot as plt  # reuse pyplot for the frozen bars
np.random.seed(19)  # fix the seed for reproducibility
trained, frozen = 0.999, 0.910  # representative lab joints: trained eyes vs random locked eyes
gap = trained - frozen  # isolate the feature-quality dividend
print('trained:', trained, 'vs frozen:', frozen, '-> gap', round(gap, 3), '(about 9 points)')  # print the ablation cost
print('frozen still far above chance 0.1: random features + trained readout learn something')  # print the weak-but-real floor
print('lesson: grow/train the encoder before buying prefix slots (K=1 already ties K=8)')  # print the design takeaway
fig, ax = plt.subplots(figsize=(6, 4))  # make a figure for the frozen comparison
ax.bar(['trained enc', 'frozen enc'], [trained, frozen], color=['#16a34a', '#6b7280'])  # draw the two bars
ax.set_ylim(0, 1.1)  # fix the accuracy range with headroom
ax.set_ylabel('joint accuracy')  # label the y-axis
ax.set_title('Frozen encoder costs about 0.089 (features, not slots)')  # title the mechanism
ax.grid(True, alpha=0.3, axis='y')  # add a light grid for reading values
plt.tight_layout()  # snug the layout
plt.show()  # render the bars


### §8 Cross-entropy — the one-number training signal

**In one sentence: cross-entropy is −log(probability the model gave the TRUE answer) — near 0 when confident-and-right, huge when confident-and-wrong — averaged over the mini-batch.**

Only the true class's probability matters; every other score counts only through softmax normalization. The log shape teaches priorities: pushing 0.9 → 0.99 earns little (0.105 → 0.010) while a confident mistake at 0.1 costs 2.303. The lab trains all five tasks with this single loss on the 1-token answer. Watch it: the curve with our two landmark dots.


In [ ]:
import numpy as np  # reuse numpy for the log curve
import matplotlib.pyplot as plt  # reuse pyplot for the loss curve
np.random.seed(19)  # fix the seed for reproducibility
p = np.linspace(0.01, 1.0, 200)  # build 200 true-class probabilities from 1% to 100%
ce = -np.log(p)  # evaluate cross-entropy -log(p) at each
print('confident+right p=0.9 -> loss', round(float(-np.log(0.9)), 3))  # print about 0.105: small
print('confident+WRONG p=0.1 -> loss', round(float(-np.log(0.1)), 3))  # print about 2.303: steep
print('lab: mean CE over the batch on the 1-token answer (multitask)')  # print how the lab aggregates it
fig, ax = plt.subplots(figsize=(6, 4))  # make a figure for the loss curve
ax.plot(p, ce, label='loss = -log(p_true)')  # draw the curve: flat near 1, steep near 0
ax.plot([0.9], [-np.log(0.9)], 'go', ms=8, label='right (p=0.9, loss 0.11)')  # mark the good point
ax.plot([0.1], [-np.log(0.1)], 'ro', ms=8, label='wrong (p=0.1, loss 2.30)')  # mark the bad point
ax.set_xlabel('probability given to the TRUE answer')  # label: only this probability counts
ax.set_ylabel('loss (lower is better)')  # label the number training minimizes
ax.set_title('CE: gentle when right, brutal when confidently wrong')  # title the moral
ax.legend()  # show which dot is which
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


## How it all fits together — the system map

One VQA answer is a pipeline with every § term at a station. **Encode** (§4): 1728 pixels → 256 summary → 4×64 photo-words. **Fuse** (§5): glue [prefix | question] into length 8, pool the question slots, score 24 answers. **Score** (§6): joint accuracy plus the two blind controls. **Ablate** (§7): freeze the eyes to price features. **Train** (§8): CE on the answer token. The map pins each term to its station; the numbers walk one toy image through all of them.


In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
import matplotlib.patches as patches  # bring in patches for the station boxes
fig, ax = plt.subplots(figsize=(11, 3.5))  # create a wide figure for the pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 12)  # fix horizontal placement range
ax.set_ylim(0, 4)  # fix vertical placement range
stages = [('image\n3x24x24', 1), ('encoder\nS4', 3), ('proj\nS4', 5), ('prefix+Q\nS5', 7), ('head 24\nS5', 9), ('CE/joint\nS6+S8', 11)]  # forward stations with section tags
ax.text(6, 3.6, 'FORWARD: pixels to answer (S4-S6)', ha='center', fontsize=9, fontweight='bold')  # forward phase banner
xc_list = [xc for _, xc in stages]  # collect the station centers for arrows
for name, xc in stages:  # draw each forward station
    ax.text(xc, 2.5, name, ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = forward pass
    _ = xc_list  # keep the comment habit inside the loop
ax.annotate('', xy=(1, 1.2), xytext=(11, 1.2), arrowprops=dict(arrowstyle='->', color='red', lw=2))  # long red back-arrow for training
ax.text(6, 0.6, 'BACKWARD (S8: CE blame) + ABLATE (S7 frozen, S6 blind controls)', ha='center', fontsize=9, fontweight='bold', color='darkred')  # red = learning/controls banner
ax.set_title('System map: image->encoder->proj->prefix+question->answer')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map


In [ ]:
import numpy as np  # reuse numpy for the end-to-end shape trace
np.random.seed(19)  # fix the seed for reproducibility
img_n = 3 * 24 * 24  # count image pixels: 3 channels of 24x24 = 1728
enc_n = 256  # record the encoder summary width from the lab
k, d, q = 4, 64, 4  # record prefix tokens, model width, question length
seq_n = k + q  # add prefix + question to get fused length 8
a = 24  # record the answer class count
print('image pixels:', img_n, '-> enc', enc_n, '-> prefix', str(k) + 'x' + str(d))  # print the vision shapes
print('fused length:', seq_n, '(prefix 4 BEFORE question 4) -> logits', a)  # print the fusion + head shapes
print('baselines: full 0.999 vs text-only 0.205 vs image-only 0.822 (color 0.0)')  # print the control numbers
print('ablation: frozen 0.91 (gap 0.089) | CE: p=0.9 -> 0.105, p=0.1 -> 2.303')  # print the training numbers
toy = np.array([0.9, 0.1, 0.2, 0.8]) @ np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.0], [0.0, 0.5]])  # replay the S4 toy projection
print('toy check: projection gives', np.round(toy, 3), '(shape-match 1.0 beats color-match 0.5)')  # print the toy verification


## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the map above (§9):

1. **Goal markdown** — the wiring sentence; check you can draw image → encoder → projection → prefix → LM → answer from memory (§4–§5).
2. **Setup** — seeds plus device line (CPU is fine); same fixed-seed habit as this lecture.
3. **Vocab + renderers + dataset** — 12 word tokens, 24 answers, five tasks (shape/color/chart/OCR/video) unified at 3×24×24; match each renderer to its question (§4).
4. **TinyVLM model** — find `encode_vis`, `self.proj`, the `torch.cat([vis, txt])` fusion line (§5), and the answer head; verify the (B, 4, 64) projection assert.
5. **Train + eval** — multitask CE (§8); `zero_vis` builds text-only (≈ 0.205) and `zero_txt` builds image-only (≈ 0.822, color 0.000) (§6).
6. **Ablations + figure** — K=1/K=8 capacity, frozen encoder (≈ 0.910) (§7), noise robustness; outputs land in `vlm.csv`, `ablations.csv`, `vlm.png`.
7. **Cleanup/next** — the handoff: today's text-only baseline becomes P20's closed-book arm.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **Where does the visual prefix sit, and why does its position matter (§5)?**

*Answer: concatenated BEFORE the question tokens ([vis | question]). Every question position attends back over every visual token, so the answer is conditioned on the image; putting it after would let early question slots answer blind.*

2. **Text-only ≈ 0.205 but image-only ≈ 0.822 joint — so is the question optional (§6)?**

*Answer: no — the joint hides a blind spot. Shape/color share images, so image-only repeats one side and scores 0.000 on color. Per-task numbers expose what the joint averages away.*

3. **K=1 nearly ties K=8 but freezing costs ≈ 9 points — where would you invest (§7)?**

*Answer: in the encoder (train it, enlarge it, raise resolution), not prefix slots. Capacity is not the bottleneck here; feature quality is — the §9 trace shows the prefix is already wide enough at 4×64.*
